In [27]:
import importlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

pd.set_option('display.max_rows', None)  
pd.set_option('display.max_columns', None)

In [15]:
# Path configuration
DATA_DIR = Path("..") / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUT_DIR = RAW_DATA_DIR / "mt1cfl_2526"
RAW_DIR = OUT_DIR / "raw_by_match"
PLAYER_PHOTOS_DIR = PROCESSED_DATA_DIR / "player_photos"
TEAM_LOGOS_DIR = PROCESSED_DATA_DIR / "team_logos"

# Create directories if they don't exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PLAYER_PHOTOS_DIR.mkdir(parents=True, exist_ok=True)
TEAM_LOGOS_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
# Load matches overview
matches_df = pd.read_csv(OUT_DIR / "matches.csv")

# Keep only essential columns
columns_to_keep = [
    'id', 'season.name', 'status.description', 'roundInfo.round', 
    'homeTeam.name', 'homeTeam.id', 'awayTeam.name', 'awayTeam.id',
    'homeScore.period1', 'homeScore.period2', 'awayScore.period1', 'awayScore.period2',
    'startTimestamp'
]

cols_to_keep = [col for col in columns_to_keep if col in matches_df.columns]
matches_df_clean = matches_df[cols_to_keep].copy()
matches_df_clean = matches_df_clean[matches_df_clean['status.description'] != 'Postponed']

print(f"Total matches (after removing postponed): {len(matches_df_clean)}")

Total matches (after removing postponed): 95


In [17]:
# Create team ID to name mapping from matches
team_mapping = {}
for _, row in matches_df_clean.iterrows():
    team_mapping[row['homeTeam.id']] = row['homeTeam.name']
    team_mapping[row['awayTeam.id']] = row['awayTeam.name']

# Create set of valid match IDs (excluding postponed matches)
valid_match_ids = set(matches_df_clean['id'].astype(str))
print(f"✅ Valid match IDs: {len(valid_match_ids)} matches (postponed matches excluded)")

# Load player match statistics from lineups.json files
player_stats_list = []
processed_matches = 0
for match_dir in RAW_DIR.glob("*"):
    # Skip postponed matches
    if match_dir.name not in valid_match_ids:
        continue
    
    lineups_file = match_dir / "lineups.json"
    if lineups_file.exists():
        try:
            with open(lineups_file, encoding='utf-8') as f:
                lineups_data = json.load(f)
            
            processed_matches += 1
            
            # Extract player statistics from both home and away teams
            for team_side in ['home', 'away']:
                if team_side in lineups_data:
                    team_data = lineups_data[team_side]
                    players = team_data.get('players', [])
                    
                    for player_entry in players:
                        player_info = player_entry.get('player', {})
                        stats = player_entry.get('statistics', {})
                        team_id = player_entry.get('teamId')
                        team_name = team_mapping.get(team_id, 'Unknown')
                        
                        # Combine player info with their statistics and team info
                        record = {
                            'match_id': match_dir.name,
                            'team_side': team_side,
                            'team_id': team_id,
                            'team_name': team_name,
                            'player_id': player_info.get('id'),
                            'player_name': player_info.get('name'),
                            'shortName': player_info.get('shortName'),
                            'position': player_info.get('position'),
                            'jerseyNumber': player_info.get('jerseyNumber'),
                            'substitute': player_entry.get('substitute', False),
                            **stats  # Unpack all statistics
                        }
                        player_stats_list.append(record)
        
        except Exception as e:
            print(f"Could not process {lineups_file}: {e}")

player_stats_df = pd.DataFrame(player_stats_list)
print(f"✅ Player stats loaded: {len(player_stats_df)} player records from {processed_matches} matches")
print(f"\n📊 Unique players: {player_stats_df['player_id'].nunique()}")
print(f"📊 Available statistics columns: {len(player_stats_df.columns)}")

✅ Valid match IDs: 95 matches (postponed matches excluded)
✅ Player stats loaded: 3782 player records from 95 matches

📊 Unique players: 285
📊 Available statistics columns: 70


In [18]:
# =============================================================================
# Scrape player current teams from Sofascore API
# =============================================================================

def get_player_current_team(player_id: int, driver) -> dict:
    """Fetch player's current team from Sofascore API."""
    try:
        url = f"https://api.sofascore.com/api/v1/player/{player_id}"
        driver.get(url)
        time.sleep(0.3)  # Small delay to avoid rate limiting
        pre_tag = driver.find_element(By.TAG_NAME, "pre")
        data = json.loads(pre_tag.text)
        
        player_data = data.get('player', {})
        team = player_data.get('team', {})
        
        return {
            'player_id': player_id,
            'current_team_id': team.get('id'),
            'current_team_name': team.get('name'),
            'retired': not bool(team.get('id'))  # If no team, likely retired
        }
    except Exception as e:
        print(f"Error fetching player {player_id}: {e}")
        return {
            'player_id': player_id,
            'current_team_id': None,
            'current_team_name': None,
            'retired': None
        }

# Check if we already have cached player team data
player_teams_cache_file = PROCESSED_DATA_DIR / "player_current_teams.json"

if player_teams_cache_file.exists():
    print("📂 Loading cached player team data...")
    with open(player_teams_cache_file, 'r', encoding='utf-8') as f:
        player_current_teams = json.load(f)
    player_current_teams = {int(k): v for k, v in player_current_teams.items()}
    print(f"✅ Loaded {len(player_current_teams)} player team mappings from cache")
else:
    print("🌐 Scraping player current teams from Sofascore...")
    print("   This will take a few minutes for all players...")
    
    # Initialize Selenium WebDriver
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
    
    try:
        unique_player_ids = player_stats_df['player_id'].unique()
        player_current_teams = {}
        
        for player_id in tqdm(unique_player_ids, desc="Fetching player teams"):
            result = get_player_current_team(int(player_id), driver)
            player_current_teams[int(player_id)] = result
        
        # Save to cache
        with open(player_teams_cache_file, 'w', encoding='utf-8') as f:
            json.dump(player_current_teams, f, indent=2)
        
        print(f"✅ Scraped and cached {len(player_current_teams)} player team mappings")
        
    finally:
        driver.quit()
        print("🔒 Closed browser")

# Create simple mappings for easy lookup
player_to_current_team = {
    pid: data.get('current_team_name') 
    for pid, data in player_current_teams.items()
}

player_to_current_team_id = {
    pid: data.get('current_team_id') 
    for pid, data in player_current_teams.items()
}

print(f"\n📊 Team status summary:")
retired_count = sum(1 for data in player_current_teams.values() if data.get('retired'))
active_count = sum(1 for data in player_current_teams.values() if not data.get('retired') and data.get('current_team_name'))
unknown_count = sum(1 for data in player_current_teams.values() if data.get('current_team_name') is None and not data.get('retired'))
print(f"   - Active players: {active_count}")
print(f"   - Retired/No team: {retired_count}")
print(f"   - Unknown status: {unknown_count}")

📂 Loading cached player team data...
✅ Loaded 285 player team mappings from cache

📊 Team status summary:
   - Active players: 285
   - Retired/No team: 0
   - Unknown status: 0


## Player Performance Analysis

Analyze player performance using actual minutes played and comprehensive statistics from lineups.json.

In [19]:
# Top 20 players by total minutes played
player_minutes_summary = player_stats_df.groupby(['player_id', 'player_name', 'position']).agg({
    'minutesPlayed': 'sum',
    'match_id': 'count',  # Number of match appearances
    'goals': 'sum',
    'goalAssist': 'sum',
    'rating': 'mean'
}).reset_index()

player_minutes_summary.columns = ['player_id', 'player_name', 'position', 'total_minutes', 'matches', 'goals', 'assists', 'avg_rating']
player_minutes_summary = player_minutes_summary.sort_values('total_minutes', ascending=False)
player_minutes_summary['avg_rating'] = player_minutes_summary['avg_rating'].round(2)

print(f"Total unique players: {len(player_minutes_summary)}")
print("=== TOP 10 PLAYERS BY MINUTES PLAYED ===\n")
display(player_minutes_summary.head(10))

Total unique players: 278
=== TOP 10 PLAYERS BY MINUTES PLAYED ===



,player_id,player_name,position,total_minutes,matches,goals,assists,avg_rating
178,1182552,Nemanja Jovičić,D,1710.0,19,0.0,0.0,6.96
103,958574,Jonathan Dresaj,D,1708.0,19,0.0,3.0,6.59
124,1005627,Aleksa Ćetković,M,1708.0,19,2.0,1.0,6.84
20,128765,Milivoje Raičević,M,1659.0,19,4.0,2.0,7.02
29,188987,Marko Kordić,G,1620.0,18,0.0,0.0,7.19
8,57363,Vladan Giljen,G,1620.0,18,0.0,0.0,7.10
16,94261,Andrija Kaluđerović,M,1620.0,18,0.0,0.0,7.12
53,800147,Bojan Zogović,G,1620.0,18,0.0,0.0,6.77
38,314142,Stefan Vico,D,1608.0,19,0.0,2.0,6.76
179,1185130,Arsenije Čepić,M,1601.0,18,0.0,0.0,6.93


## Organized Data Export

This section creates a clean, organized data structure with:
1. **players_metadata.csv** - Static player info (name, position, height, country, etc.)
2. **teams_metadata.csv** - Static team info (name, colors, country)
3. **matches_metadata.csv** - Match info (teams, scores, timestamp, round)
4. **match_player_statistics.csv** - Per-player per-match stats
5. **match_team_statistics.csv** - Per-team per-match stats

This eliminates the need to iterate over raw data for common analysis tasks.

In [20]:
# =============================================================================
# 1. TEAMS METADATA - Extract static team information
# =============================================================================

teams_data = {}

# Extract from matches
for _, row in matches_df.iterrows():
    # Home team
    if row['homeTeam.id'] not in teams_data:
        teams_data[row['homeTeam.id']] = {
            'team_id': row['homeTeam.id'],
            'team_name': row['homeTeam.name']
        }
    # Away team
    if row['awayTeam.id'] not in teams_data:
        teams_data[row['awayTeam.id']] = {
            'team_id': row['awayTeam.id'],
            'team_name': row['awayTeam.name']
        }

teams_meta = pd.DataFrame(list(teams_data.values()))
teams_output_path = PROCESSED_DATA_DIR / "teams_metadata.csv"
teams_meta.to_csv(teams_output_path, index=False)

print(f"✅ Saved {len(teams_meta)} teams to {teams_output_path}")
display(teams_meta)

✅ Saved 10 teams to ..\data\processed\teams_metadata.csv


,team_id,team_name
0,37956,FK Arsenal Tivat
1,6219,FK Jedinstvo Bijelo Polje
2,6216,OFK Petrovac
3,7581,FK Bokelj Kotor
4,6226,FK Dečić Tuzi
5,327162,Mladost DG
6,24312,FK Jezero
7,5143,FK Budućnost Podgorica
8,6224,FK Sutjeska Nikšić
9,36019,FK Mornar Bar


In [21]:
# =============================================================================
# 2. PLAYERS METADATA - Extract STATIC player information (no match stats)
# =============================================================================

players_static_data = {}

for match_dir in tqdm(RAW_DIR.glob("*"), desc="Extracting player metadata"):
    if match_dir.name not in valid_match_ids:
        continue
    
    lineups_file = match_dir / "lineups.json"
    if lineups_file.exists():
        try:
            with open(lineups_file, encoding='utf-8') as f:
                lineups_data = json.load(f)
            
            for team_side in ['home', 'away']:
                if team_side in lineups_data:
                    for player_entry in lineups_data[team_side].get('players', []):
                        player_info = player_entry.get('player', {})
                        player_id = player_info.get('id')
                        
                        if player_id and player_id not in players_static_data:
                            # Extract country info from nested structure
                            country = player_info.get('country', {})
                            
                            # Extract market value from proposedMarketValueRaw
                            proposed_market_value = player_info.get('proposedMarketValueRaw', {})
                            market_value = proposed_market_value.get('value') if proposed_market_value else None
                            market_value_currency = proposed_market_value.get('currency', '') if proposed_market_value else player_info.get('marketValueCurrency', '')
                            
                            players_static_data[player_id] = {
                                'id': player_id,
                                'name': player_info.get('name', ''),
                                'shortName': player_info.get('shortName', ''),
                                'position': player_info.get('position', ''),
                                'height': player_info.get('height'),
                                'country_name': country.get('name', '') if country else '',
                                'dateOfBirthTimestamp': player_info.get('dateOfBirthTimestamp'),
                                'marketValue': market_value
                            }
        except Exception as e:
            print(f"Could not process {lineups_file}: {e}")

# Convert to DataFrame
players_static = pd.DataFrame(list(players_static_data.values()))

# Convert timestamp to readable date
def timestamp_to_date(ts):
    if pd.isna(ts) or ts is None:
        return None
    try:
        return pd.to_datetime(ts, unit='s').strftime('%Y-%m-%d')
    except:
        return None

players_static['dateOfBirth'] = players_static['dateOfBirthTimestamp'].apply(timestamp_to_date)
players_static = players_static.drop(columns=['dateOfBirthTimestamp'])

# Add has_player_photo flag
players_static['has_player_photo'] = players_static['id'].apply(
    lambda pid: (PLAYER_PHOTOS_DIR / f"{pid}.png").exists()
)

# Save
players_output_path = PROCESSED_DATA_DIR / "players_metadata.csv"
players_static.to_csv(players_output_path, index=False)

print(f"\n✅ Saved {len(players_static)} players to {players_output_path}")
display(players_static.head(10))

Extracting player metadata: 97it [00:00, 853.43it/s]


✅ Saved 285 players to ..\data\processed\players_metadata.csv


,id,name,shortName,position,height,country_name,marketValue,dateOfBirth,has_player_photo
0,927881,Stefan Spasojević,S. Spasojević,G,193.0,Montenegro,72000.0,1993-08-23,True
1,314142,Stefan Vico,S. Vico,D,177.0,Montenegro,110000.0,1995-02-28,True
2,94246,Igor Ćuković,I. Ćuković,D,191.0,Montenegro,24000.0,1993-06-06,True
3,57463,Saša Balić,S. Balić,D,184.0,Montenegro,NaN,1990-01-29,True
4,909420,Marko Čavor,M. Čavor,D,175.0,Montenegro,170000.0,1999-07-05,True
5,1381377,Kristijan Radunović,K. Radunović,M,184.0,Montenegro,72000.0,2004-09-08,False
6,1486746,Alhassan Baba Musah,A. B. Musah,M,176.0,Ghana,72000.0,2000-03-08,True
7,807177,Žarko Vilotijević,Ž. Vilotijević,M,180.0,Montenegro,73000.0,1996-01-04,False
8,572018,Velizar Janketić,V. Janketić,M,175.0,Montenegro,54000.0,1996-11-15,True
9,813112,Luka Maraš,L. Maraš,D,NaN,Montenegro,73000.0,1996-05-24,True


In [22]:
# =============================================================================
# 3. MATCHES METADATA - Clean match information
# =============================================================================

matches_meta = matches_df_clean[['id', 'season.name', 'status.description', 'roundInfo.round',
                                 'homeTeam.name', 'homeTeam.id', 'awayTeam.name', 'awayTeam.id',
                                 'homeScore.period1', 'homeScore.period2', 'awayScore.period1', 'awayScore.period2',
                                 'startTimestamp']].copy()

# Rename columns for clarity
matches_meta.columns = [
    'match_id', 'season', 'status', 'round',
    'homeTeam_name', 'homeTeam_id', 'awayTeam_name', 'awayTeam_id',
    'homeScore_period1', 'homeScore_period2', 'awayScore_period1', 'awayScore_period2',
    'startTimestamp'
]

# Add total scores
matches_meta['homeScore_total'] = matches_meta['homeScore_period1'].fillna(0) + matches_meta['homeScore_period2'].fillna(0)
matches_meta['awayScore_total'] = matches_meta['awayScore_period1'].fillna(0) + matches_meta['awayScore_period2'].fillna(0)

# Convert timestamp to readable datetime
matches_meta['match_datetime'] = pd.to_datetime(matches_meta['startTimestamp'], unit='s')
matches_meta['match_date'] = matches_meta['match_datetime'].dt.strftime('%Y-%m-%d')

# Define the set of matches we expect formations for (postponed already removed)
matches_with_lineups = set(matches_meta['match_id'].astype(str))

# Collect formations from lineups (skip postponed matches already filtered out)
formation_records = []
missing_formations = []

for match_dir in tqdm(RAW_DIR.glob("*"), desc="Collecting formations"):
    match_id = match_dir.name
    if match_id not in matches_with_lineups:
        continue

    lineups_file = match_dir / "lineups.json"
    if not lineups_file.exists():
        continue

    try:
        with open(lineups_file, encoding='utf-8') as f:
            lineups_data = json.load(f)
    except Exception as e:
        print(f"⚠️ Could not read formations for match {match_id}: {e}")
        continue

    home_formation = lineups_data.get('home', {}).get('formation')
    away_formation = lineups_data.get('away', {}).get('formation')

    if not home_formation or not away_formation:
        missing_formations.append(int(match_id))
        continue

    formation_records.append({
        'match_id': int(match_id),
        'homeTeam_formation': home_formation,
        'awayTeam_formation': away_formation,
    })

formations_df = pd.DataFrame(formation_records).drop_duplicates(subset=['match_id'], keep='last')

if formations_df.empty:
    raise ValueError("No formation data collected. Please verify the raw lineups files.")

if missing_formations:
    print(f"⚠️ Skipped {len(set(missing_formations))} matches without formation data: {sorted(set(missing_formations))}")

# Merge formations into metadata
matches_meta = matches_meta.merge(formations_df, on='match_id', how='left')

# Reorder columns
matches_meta = matches_meta[[
    'match_id', 'round', 'match_date', 'match_datetime', 'status',
    'homeTeam_id', 'homeTeam_name', 'homeTeam_formation',
    'awayTeam_id', 'awayTeam_name', 'awayTeam_formation',
    'homeScore_period1', 'homeScore_period2', 'homeScore_total',
    'awayScore_period1', 'awayScore_period2', 'awayScore_total'
]]

# Save
matches_output_path = PROCESSED_DATA_DIR / "matches_metadata.csv"
matches_meta.to_csv(matches_output_path, index=False)

print(f"✅ Saved {len(matches_meta)} matches to {matches_output_path}")
display(matches_meta.head(10))

✅ Saved 95 matches to ..\data\processed\matches_metadata.csv


,match_id,round,match_date,match_datetime,status,homeTeam_id,homeTeam_name,homeTeam_formation,awayTeam_id,awayTeam_name,awayTeam_formation,homeScore_period1,homeScore_period2,homeScore_total,awayScore_period1,awayScore_period2,awayScore_total
0,14147310,1,2025-08-04,2025-08-04 18:00:00,Ended,37956,FK Arsenal Tivat,3-4-2-1,6219,FK Jedinstvo Bijelo Polje,4-2-3-1,1.0,1.0,2.0,0.0,0.0,0.0
1,14147408,1,2025-08-04,2025-08-04 18:00:00,Ended,6216,OFK Petrovac,4-4-2,7581,FK Bokelj Kotor,4-2-3-1,1.0,1.0,2.0,0.0,0.0,0.0
2,14147300,1,2025-08-04,2025-08-04 18:00:00,Ended,6226,FK Dečić Tuzi,4-1-4-1,327162,Mladost DG,4-2-3-1,2.0,1.0,3.0,0.0,2.0,2.0
3,14147319,1,2025-08-04,2025-08-04 18:00:00,Ended,24312,FK Jezero,4-4-2,5143,FK Budućnost Podgorica,4-2-3-1,1.0,0.0,1.0,0.0,2.0,2.0
4,14147314,1,2025-08-04,2025-08-04 19:00:00,Ended,6224,FK Sutjeska Nikšić,4-2-3-1,36019,FK Mornar Bar,4-2-3-1,0.0,3.0,3.0,0.0,0.0,0.0
5,14147315,2,2025-08-10,2025-08-10 18:00:00,Ended,36019,FK Mornar Bar,4-2-3-1,37956,FK Arsenal Tivat,5-4-1,0.0,2.0,2.0,2.0,0.0,2.0
6,14147309,2,2025-08-10,2025-08-10 18:00:00,Ended,5143,FK Budućnost Podgorica,3-4-3,6224,FK Sutjeska Nikšić,4-2-3-1,0.0,0.0,0.0,0.0,1.0,1.0
7,14147297,2,2025-08-10,2025-08-10 18:00:00,Ended,7581,FK Bokelj Kotor,4-1-4-1,327162,Mladost DG,4-4-2,0.0,2.0,2.0,1.0,1.0,2.0
8,14147301,2,2025-08-10,2025-08-10 18:00:00,Ended,6216,OFK Petrovac,4-2-3-1,24312,FK Jezero,4-2-3-1,0.0,1.0,1.0,0.0,1.0,1.0
9,14147313,2,2025-08-10,2025-08-10 19:00:00,Ended,6219,FK Jedinstvo Bijelo Polje,4-3-2-1,6226,FK Dečić Tuzi,4-2-3-1,0.0,0.0,0.0,0.0,1.0,1.0


In [23]:
# =============================================================================
# 4. MATCH PLAYER STATISTICS - Complete per-match per-player statistics
# =============================================================================

match_player_stats_list = []
dribbles_list = []

for match_dir in tqdm([d for d in RAW_DIR.glob("*") if d.name in valid_match_ids], desc="Loading player stats"):
    with open(match_dir / "lineups.json", encoding='utf-8') as f:
        lineups_data = json.load(f)
    
    # Get match info for team IDs and names
    match_info = matches_df_clean[matches_df_clean['id'] == int(match_dir.name)].iloc[0]
    team_ids = {
        'home': int(match_info['homeTeam.id']),
        'away': int(match_info['awayTeam.id'])
    }
    team_names = {
        'home': match_info['homeTeam.name'],
        'away': match_info['awayTeam.name']
    }
    
    for team_side in ['home', 'away']:
        team_data = lineups_data[team_side]
        
        for player_entry in team_data['players']:
            player_info = player_entry['player']
            stats = player_entry.get('statistics', {})
            
            record = {
                'match_id': int(match_dir.name),
                'player_id': player_info['id'],
                'player_name': player_info.get('name', ''),
                'team_id': team_ids[team_side],
                'team_name': team_names[team_side],
                'team_side': team_side,
                'position_played': player_entry.get('position', player_info.get('position', '')),
                'shirtNumber': player_entry.get('shirtNumber'),
                'substitute': player_entry.get('substitute', False),
                'played': stats.get('minutesPlayed', 0) > 0,
                **{k: v for k, v in stats.items() if k not in ['ratingVersions', 'statisticsType']}
            }
            match_player_stats_list.append(record)
    
    # Load dribbles data
    stats_tabs_file = match_dir / "player_stats_tabs.csv"
    if stats_tabs_file.exists():
        stats_tabs_df = pd.read_csv(stats_tabs_file)
        dribbles_df = stats_tabs_df[
            (stats_tabs_df['metric_key'] == 'totalContest') & 
            (stats_tabs_df['metric'] == 'Dribbles (successful)')
        ][['match_id', 'player_id', 'value', 'secondary']].rename(
            columns={'value': 'totalDribble', 'secondary': 'successfulDribbles'}
        )
        if len(dribbles_df) > 0:
            dribbles_list.append(dribbles_df)

# Create DataFrame and merge dribbles
match_player_stats = pd.DataFrame(match_player_stats_list)

if dribbles_list:
    dribbles_data = pd.concat(dribbles_list, ignore_index=True).drop_duplicates(subset=['match_id', 'player_id'])
    match_player_stats = match_player_stats.drop(columns=['successfulDribbles', 'totalDribble'], errors='ignore').merge(
        dribbles_data, on=['match_id', 'player_id'], how='left'
    )
    print(f"✅ Merged dribbles data for {len(dribbles_data)} player-match records")

match_player_stats[['successfulDribbles', 'totalDribble']] = match_player_stats[['successfulDribbles', 'totalDribble']].fillna(0)

# Add player's current team from scraped data (both ID and name)
match_player_stats['player_current_team_id'] = match_player_stats['player_id'].map(player_to_current_team_id)
match_player_stats['player_current_team'] = match_player_stats['player_id'].map(player_to_current_team)

# Reorder columns
id_cols = ['match_id', 'player_id', 'player_name', 'team_id', 'team_name', 'player_current_team_id', 'player_current_team', 'team_side', 'position_played', 'shirtNumber', 'substitute', 'played']
stat_cols = [col for col in match_player_stats.columns if col not in id_cols]
match_player_stats = match_player_stats[id_cols + sorted(stat_cols)]

# Save
match_player_stats.to_csv(PROCESSED_DATA_DIR / "match_player_statistics.csv", index=False)

print(f"✅ Saved {len(match_player_stats)} player-match records")
print(f"   - Statistics columns: {len(stat_cols)}")
display(match_player_stats.head(5))

Loading player stats: 100%|██████████| 95/95 [00:01<00:00, 88.65it/s]


✅ Merged dribbles data for 3782 player-match records
✅ Saved 3782 player-match records
   - Statistics columns: 60


,match_id,player_id,player_name,team_id,team_name,player_current_team_id,player_current_team,team_side,position_played,shirtNumber,substitute,played,accurateCross,accurateKeeperSweeper,accurateLongBalls,accurateOppositionHalfPasses,accurateOwnHalfPasses,accuratePass,aerialLost,aerialWon,ballRecovery,bigChanceCreated,bigChanceMissed,blockedScoringAttempt,challengeLost,clearanceOffLine,crossNotClaimed,dispossessed,duelLost,duelWon,errorLeadToAShot,expectedAssists,expectedGoals,fouls,goalAssist,goals,goodHighClaim,hitWoodwork,interceptionWon,keyPass,minutesPlayed,onTargetScoringAttempt,outfielderBlock,ownGoals,penaltyConceded,penaltyFaced,penaltyMiss,penaltySave,penaltyWon,possessionLostCtrl,rating,savedShotsFromInsideTheBox,saves,shotOffTarget,successfulDribbles,totalClearance,totalContest,totalCross,totalDribble,totalKeeperSweeper,totalLongBalls,totalOffside,totalOppositionHalfPasses,totalOwnHalfPasses,totalPass,totalShots,totalTackle,touches,unsuccessfulTouch,wasFouled,wonContest,wonTackle
0,14147297,927881,Stefan Spasojević,7581,FK Bokelj Kotor,7581,FK Bokelj Kotor,home,G,88,False,True,0.0,0.0,6.0,2.0,20.0,22.0,0.0,0.0,5.0,NaN,NaN,NaN,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,90.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,3.0,6.4,NaN,2.0,NaN,0.0,1.0,0.0,0.0,0.0,2.0,9.0,0.0,3.0,22.0,26.0,0.0,0.0,27.0,0.0,0.0,0.0,0.0
1,14147297,314142,Stefan Vico,7581,FK Bokelj Kotor,7581,FK Bokelj Kotor,home,D,22,False,True,0.0,NaN,1.0,4.0,6.0,10.0,2.0,1.0,3.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,46.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,2.0,6.4,NaN,NaN,NaN,0.0,0.0,0.0,1.0,0.0,NaN,1.0,0.0,5.0,6.0,11.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0
2,14147297,94246,Igor Ćuković,7581,FK Bokelj Kotor,7581,FK Bokelj Kotor,home,D,26,False,True,0.0,NaN,3.0,7.0,20.0,27.0,0.0,2.0,6.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,2.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,59.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,7.0,6.5,NaN,NaN,NaN,0.0,0.0,0.0,1.0,0.0,NaN,5.0,0.0,12.0,22.0,34.0,0.0,0.0,38.0,0.0,0.0,0.0,0.0
3,14147297,57463,Saša Balić,7581,FK Bokelj Kotor,241802,No team,home,D,34,False,True,0.0,NaN,5.0,8.0,36.0,45.0,1.0,2.0,10.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,6.0,2.0,0.0,0.0,0.0,3.0,0.0,0.0,NaN,NaN,1.0,NaN,90.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,13.0,6.3,NaN,NaN,NaN,0.0,4.0,0.0,0.0,0.0,NaN,10.0,0.0,15.0,41.0,57.0,0.0,0.0,64.0,0.0,1.0,0.0,0.0
4,14147297,909420,Marko Čavor,7581,FK Bokelj Kotor,7581,FK Bokelj Kotor,home,D,29,False,True,0.0,NaN,1.0,11.0,8.0,19.0,0.0,2.0,4.0,NaN,NaN,1.0,0.0,NaN,NaN,NaN,5.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,1.0,90.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,11.0,6.4,NaN,NaN,NaN,2.0,0.0,3.0,1.0,3.0,NaN,5.0,0.0,19.0,11.0,30.0,1.0,0.0,42.0,0.0,1.0,2.0,0.0


In [24]:
# =============================================================================
# 5. MATCH TEAM STATISTICS - Team-level per-match statistics from statistics.json
# =============================================================================

match_team_stats_list = []

for match_dir in tqdm(RAW_DIR.glob("*"), desc="Loading team match stats"):
    if match_dir.name not in valid_match_ids:
        continue
    
    stats_file = match_dir / "statistics.json"
    if stats_file.exists():
        try:
            with open(stats_file, encoding='utf-8') as f:
                stats_data = json.load(f)
            
            # Get match info to map team IDs
            match_info = matches_df_clean[matches_df_clean['id'] == int(match_dir.name)]
            if len(match_info) == 0:
                continue
            match_info = match_info.iloc[0]
            
            home_record = {
                'match_id': int(match_dir.name),
                'team_id': int(match_info['homeTeam.id']),
                'team_name': match_info['homeTeam.name'],
                'team_side': 'home',
            }
            away_record = {
                'match_id': int(match_dir.name),
                'team_id': int(match_info['awayTeam.id']),
                'team_name': match_info['awayTeam.name'],
                'team_side': 'away',
            }
            
            # Parse statistics
            for stat_section in stats_data.get('statistics', []):
                if stat_section.get('period') == 'ALL':
                    for group in stat_section.get('groups', []):
                        for item in group.get('statisticsItems', []):
                            key = item.get('key', item.get('name', '').replace(' ', '_').lower())
                            home_record[key] = item.get('homeValue')
                            away_record[key] = item.get('awayValue')
            
            match_team_stats_list.append(home_record)
            match_team_stats_list.append(away_record)
        
        except Exception as e:
            print(f"Could not process {stats_file}: {e}")

# Convert to DataFrame
match_team_stats = pd.DataFrame(match_team_stats_list)

# Reorder columns
id_cols = ['match_id', 'team_id', 'team_name', 'team_side']
stat_cols = [col for col in match_team_stats.columns if col not in id_cols]
match_team_stats = match_team_stats[id_cols + sorted(stat_cols)]

# Save
team_stats_output_path = PROCESSED_DATA_DIR / "match_team_statistics.csv"
match_team_stats.to_csv(team_stats_output_path, index=False)

print(f"✅ Saved {len(match_team_stats)} team-match records to {team_stats_output_path}")
display(match_team_stats.head(10))

Loading team match stats: 97it [00:00, 1341.41it/s]

✅ Saved 190 team-match records to ..\data\processed\match_team_statistics.csv


,match_id,team_id,team_name,team_side,accurateCross,accurateLongBalls,accuratePasses,accurateThroughBall,aerialDuelsPercentage,ballPossession,blockedScoringAttempt,cornerKicks,dispossessed,dribblesPercentage,duelWonPercent,expectedGoals,fouls,freeKicks,goalKicks,goalkeeperSaves,groundDuelsPercentage,hitWoodwork,interceptionWon,offsides,passes,redCards,shotsOffGoal,shotsOnGoal,throwIns,totalClearance,totalShotsInsideBox,totalShotsOnGoal,totalShotsOutsideBox,totalTackle,yellowCards
0,14147297,7581,FK Bokelj Kotor,home,4,25,219,0,23,46,3,4,12,6,34,1.12,14,3,10,2,26,1,5,3,310,0,2,3,32,13,4,9,5,5,4
1,14147297,327162,Mladost DG,away,3,20,315,0,16,54,2,5,6,7,40,1.35,18,4,6,1,40,0,19,1,388,0,5,4,23,14,7,11,4,3,5
2,14147298,6226,FK Dečić Tuzi,home,3,32,366,1,10,50,0,3,9,8,33,1.31,12,3,11,2,44,0,19,4,435,0,7,4,22,11,6,11,5,3,1
3,14147298,36019,FK Mornar Bar,away,4,18,301,1,20,50,5,4,11,19,42,1.74,16,1,8,2,48,0,9,2,383,0,3,3,22,8,9,11,2,0,2
4,14147299,5143,FK Budućnost Podgorica,home,3,21,272,1,4,47,9,8,4,21,37,1.79,11,0,9,2,45,1,11,4,342,0,5,5,27,13,11,20,9,4,0
5,14147299,6226,FK Dečić Tuzi,away,8,23,329,0,11,53,4,8,5,11,30,0.59,8,3,10,3,31,0,9,3,386,0,4,2,13,13,4,10,6,2,1
6,14147300,6226,FK Dečić Tuzi,home,1,34,327,1,11,49,0,2,5,10,30,0.30,16,0,10,2,25,0,15,3,417,0,1,3,28,19,4,4,0,4,3
7,14147300,327162,Mladost DG,away,6,26,395,0,9,51,2,3,4,19,41,0.61,12,2,4,0,46,0,14,4,482,0,6,4,26,12,4,12,8,6,1
8,14147301,6216,OFK Petrovac,home,5,33,396,1,17,54,3,3,7,16,44,2.61,10,5,7,1,48,1,19,2,472,0,8,6,26,14,11,18,7,5,1
9,14147301,24312,FK Jezero,away,0,29,334,3,10,46,2,2,2,12,28,0.70,17,1,14,5,28,0,19,1,409,0,5,2,19,28,4,9,5,5,1


In [25]:
# =============================================================================
# 7. BONUS: Aggregated Player Season Statistics (convenience file)
# =============================================================================

# Create aggregated player stats for quick access to season totals
player_season_stats = match_player_stats[match_player_stats['played'] == True].groupby('player_id').agg({
    # Team info (take last/most recent)
    'team_id': 'last',
    'team_name': 'last',
    # Counts
    'match_id': 'count',  # appearances
    # Time
    'minutesPlayed': 'sum',
    # Attacking
    'goals': 'sum',
    'goalAssist': 'sum',
    'expectedGoals': 'sum',
    'expectedAssists': 'sum',
    'totalShots': 'sum',
    'onTargetScoringAttempt': 'sum',
    'keyPass': 'sum',
    'bigChanceCreated': 'sum',
    # Passing
    'totalPass': 'sum',
    'accuratePass': 'sum',
    'totalLongBalls': 'sum',
    'accurateLongBalls': 'sum',
    'totalCross': 'sum',
    'accurateCross': 'sum',
    # Defending
    'totalTackle': 'sum',
    'wonTackle': 'sum',
    'interceptionWon': 'sum',
    'totalClearance': 'sum',
    'ballRecovery': 'sum',
    'outfielderBlock': 'sum',
    # Duels
    'duelWon': 'sum',
    'duelLost': 'sum',
    'aerialWon': 'sum',
    'aerialLost': 'sum',
    # Discipline
    'fouls': 'sum',
    'wasFouled': 'sum',
    'successfulDribbles': 'sum',
    'totalDribble': 'sum',
    # Rating
    'rating': 'mean',
}).reset_index()

# Rename columns
player_season_stats.columns = [
    'player_id', 'team_id', 'team_name', 'appearances',
    'minutes_played', 'goals', 'assists', 'xG', 'xA',
    'total_shots', 'shots_on_target', 'key_passes', 'big_chances_created',
    'total_passes', 'accurate_passes', 'total_long_balls', 'accurate_long_balls',
    'total_crosses', 'accurate_crosses',
    'total_tackles', 'tackles_won', 'interceptions', 'clearances', 'ball_recoveries', 'blocks',
    'duels_won', 'duels_lost', 'aerial_won', 'aerial_lost',
    'fouls_committed', 'fouls_won',
    'dribbles_successful', 'dribbles_total',
    'avg_rating'
]

# Add derived metrics
player_season_stats['minutes_per_game'] = (player_season_stats['minutes_played'] / player_season_stats['appearances']).round(1)
player_season_stats['pass_accuracy'] = ((player_season_stats['accurate_passes'] / player_season_stats['total_passes']) * 100).round(1)
player_season_stats['shot_accuracy'] = ((player_season_stats['shots_on_target'] / player_season_stats['total_shots'].replace(0, np.nan)) * 100).round(1)
player_season_stats['dribble_success_rate'] = ((player_season_stats['dribbles_successful'] / player_season_stats['dribbles_total'].replace(0, np.nan)) * 100).round(1)
player_season_stats['duel_success_rate'] = ((player_season_stats['duels_won'] / (player_season_stats['duels_won'] + player_season_stats['duels_lost']).replace(0, np.nan)) * 100).round(1)
player_season_stats['goals_per_90'] = ((player_season_stats['goals'] / player_season_stats['minutes_played']) * 90).round(2)
player_season_stats['assists_per_90'] = ((player_season_stats['assists'] / player_season_stats['minutes_played']) * 90).round(2)
player_season_stats['avg_rating'] = player_season_stats['avg_rating'].round(2)

# Merge with static player info
player_season_stats = player_season_stats.merge(
    players_static[['id', 'name', 'shortName', 'position', 'country_name', 'dateOfBirth', 'has_player_photo']],
    left_on='player_id',
    right_on='id',
    how='left'
).drop(columns=['id'])

# Reorder columns
first_cols = ['player_id', 'name', 'shortName', 'position', 'team_name', 'country_name', 'dateOfBirth']
other_cols = [c for c in player_season_stats.columns if c not in first_cols]
player_season_stats = player_season_stats[first_cols + other_cols]

# Sort by minutes played
player_season_stats = player_season_stats.sort_values('minutes_played', ascending=False).reset_index(drop=True)

# Save
season_stats_output_path = PROCESSED_DATA_DIR / "player_season_statistics.csv"
player_season_stats.to_csv(season_stats_output_path, index=False)

print(f"✅ Saved {len(player_season_stats)} aggregated player records to {season_stats_output_path}")
print(f"\n🏆 Top 10 players by minutes played:")
player_season_stats.head(5)

✅ Saved 253 aggregated player records to ..\data\processed\player_season_statistics.csv

🏆 Top 10 players by minutes played:


,player_id,name,shortName,position,team_name,country_name,dateOfBirth,team_id,appearances,minutes_played,goals,assists,xG,xA,total_shots,shots_on_target,key_passes,big_chances_created,total_passes,accurate_passes,total_long_balls,accurate_long_balls,total_crosses,accurate_crosses,total_tackles,tackles_won,interceptions,clearances,ball_recoveries,blocks,duels_won,duels_lost,aerial_won,aerial_lost,fouls_committed,fouls_won,dribbles_successful,dribbles_total,avg_rating,minutes_per_game,pass_accuracy,shot_accuracy,dribble_success_rate,duel_success_rate,goals_per_90,assists_per_90,has_player_photo
0,1182552,Nemanja Jovičić,N. Jovičić,D,FK Jezero,Serbia,2000-01-01,24312,19,1710.0,0.0,0.0,0.22,0.00,4.0,0.0,2.0,0.0,1053.0,903.0,161.0,77.0,1.0,0.0,8.0,4.0,52.0,79.0,262.0,11.0,83.0,127.0,44.0,32.0,27.0,8.0,5.0,6.0,6.96,90.0,85.8,0.0,83.3,39.5,0.00,0.00,True
1,958574,Jonathan Dresaj,J. Dresaj,D,FK Dečić Tuzi,Montenegro,2000-03-15,6226,19,1708.0,0.0,3.0,0.38,2.03,8.0,2.0,17.0,2.0,644.0,497.0,81.0,42.0,63.0,26.0,2.0,0.0,28.0,35.0,152.0,3.0,78.0,195.0,14.0,30.0,11.0,11.0,32.0,45.0,6.59,89.9,77.2,25.0,71.1,28.6,0.00,0.16,True
2,1005627,Aleksa Ćetković,A. Ćetković,M,FK Arsenal Tivat,Montenegro,2004-02-13,37956,19,1708.0,2.0,1.0,1.25,0.21,12.0,3.0,7.0,0.0,656.0,484.0,89.0,47.0,13.0,4.0,1.0,0.0,35.0,30.0,207.0,0.0,148.0,243.0,55.0,54.0,32.0,31.0,14.0,18.0,6.84,89.9,73.8,25.0,77.8,37.9,0.11,0.05,True
3,128765,Milivoje Raičević,M. Raičević,M,FK Jezero,Montenegro,1993-07-21,24312,19,1659.0,4.0,2.0,3.80,1.88,35.0,14.0,23.0,1.0,648.0,504.0,22.0,7.0,8.0,0.0,9.0,7.0,20.0,14.0,105.0,1.0,123.0,237.0,37.0,52.0,25.0,25.0,24.0,55.0,7.02,87.3,77.8,40.0,43.6,34.2,0.22,0.11,True
4,94261,Andrija Kaluđerović,A. Kaluđeriović,M,FK Mornar Bar,Montenegro,1993-10-29,36019,18,1620.0,0.0,0.0,0.19,0.09,9.0,2.0,6.0,0.0,886.0,763.0,87.0,61.0,9.0,3.0,50.0,32.0,30.0,17.0,245.0,3.0,143.0,187.0,18.0,20.0,32.0,27.0,13.0,18.0,7.12,90.0,86.1,22.2,72.2,43.3,0.00,0.00,True


In [26]:
player_season_stats.sort_values('tackles_won', ascending=False).head(5)

,player_id,name,shortName,position,team_name,country_name,dateOfBirth,team_id,appearances,minutes_played,goals,assists,xG,xA,total_shots,shots_on_target,key_passes,big_chances_created,total_passes,accurate_passes,total_long_balls,accurate_long_balls,total_crosses,accurate_crosses,total_tackles,tackles_won,interceptions,clearances,ball_recoveries,blocks,duels_won,duels_lost,aerial_won,aerial_lost,fouls_committed,fouls_won,dribbles_successful,dribbles_total,avg_rating,minutes_per_game,pass_accuracy,shot_accuracy,dribble_success_rate,duel_success_rate,goals_per_90,assists_per_90,has_player_photo
4,94261,Andrija Kaluđerović,A. Kaluđeriović,M,FK Mornar Bar,Montenegro,1993-10-29,36019,18,1620.0,0.0,0.0,0.19,0.09,9.0,2.0,6.0,0.0,886.0,763.0,87.0,61.0,9.0,3.0,50.0,32.0,30.0,17.0,245.0,3.0,143.0,187.0,18.0,20.0,32.0,27.0,13.0,18.0,7.12,90.0,86.1,22.2,72.2,43.3,0.00,0.00,True
20,1536333,Stefan Radojevic,S. Radojevic,M,FK Jedinstvo Bijelo Polje,Montenegro,2005-03-04,6219,18,1459.0,1.0,0.0,1.59,0.18,23.0,10.0,6.0,0.0,375.0,242.0,38.0,9.0,7.0,1.0,28.0,17.0,32.0,24.0,123.0,4.0,82.0,188.0,23.0,33.0,34.0,12.0,4.0,16.0,6.67,81.1,64.5,43.5,25.0,30.4,0.06,0.00,True
47,1423517,Lazar Maraš,L. Maraš,D,FK Dečić Tuzi,Montenegro,2006-01-01,6226,17,1318.0,0.0,1.0,0.34,1.69,5.0,1.0,9.0,2.0,538.0,416.0,56.0,30.0,38.0,15.0,19.0,15.0,28.0,28.0,149.0,6.0,70.0,148.0,19.0,17.0,21.0,9.0,10.0,19.0,6.75,77.5,77.3,20.0,52.6,32.1,0.00,0.07,True
42,813112,Luka Maraš,L. Maraš,D,FK Bokelj Kotor,Montenegro,1996-05-24,7581,18,1339.0,2.0,1.0,1.60,2.02,12.0,7.0,14.0,1.0,304.0,215.0,39.0,15.0,46.0,15.0,17.0,11.0,9.0,27.0,83.0,2.0,111.0,182.0,11.0,27.0,28.0,40.0,35.0,43.0,6.89,74.4,70.7,58.3,81.4,37.9,0.13,0.07,True
48,889063,Stefan Fićović,S. Fićović,M,OFK Petrovac,Serbia,1998-05-31,6216,16,1291.0,2.0,2.0,0.60,0.23,12.0,2.0,8.0,0.0,565.0,462.0,52.0,19.0,7.0,1.0,15.0,11.0,29.0,19.0,144.0,5.0,93.0,150.0,28.0,27.0,32.0,15.0,11.0,14.0,6.86,80.7,81.8,16.7,78.6,38.3,0.14,0.14,True
